# Replay buffer experiment

Run from the repository root with the model and task data described in [the README](README.md). The final cell launches the full training and evaluation sweep.

In [ ]:
from huggingface_hub import login
login()

## Evaluation

Evaluate code generation on HumanEval and report progress for each task.

In [ ]:
import argparse
import math

import torch
import tqdm

import eval_model
from get_datasets import get_dataset, get_batch_iterator

def evaluate_tqdm_total(ds_name, model, tokenizer, args):
    real_ds = "openai_humaneval" if ds_name == "codealpaca" else ds_name
    if real_ds == "openai_humaneval":
        n_ex, split = 164, "test"
    else:
        split = "test"
        try:                                     # most tasks have a test split
            n_ex = len(get_dataset(real_ds, split=split, silent=True, data_fraction=1.0))
        except Exception:                        # fallback → train
            split = "train"
            n_ex  = len(get_dataset(real_ds, split=split, silent=True, data_fraction=1.0))

    n_batches = math.ceil(n_ex / args.batch_size)
    loader    = get_batch_iterator(
        names=[real_ds], tokenizer=tokenizer, shuffle=False,
        max_length=2048, max_prompt_length=2048,  # keep prompts intact
        sft_mode=True, split=split, n_epochs=1, batch_size=args.batch_size,
        cache_dir=None, seed=args.seed, data_fraction=1.0,
    )

    gen_kwargs = dict(
        max_new_tokens=512, do_sample=args.sample,
        repetition_penalty=1.0, length_penalty=1.0,
        use_cache=True, pad_token_id=model.config.pad_token_id,
    )
    if args.sample:
        gen_kwargs.update(temperature=0.6, top_k=50, top_p=0.9)

    preds, gold = [], []
    pbar = tqdm.tqdm(enumerate(loader, 1), total=n_batches,
                     desc=f"{ds_name} → {real_ds} ({split})", unit="batch", leave=False)

    for idx, batch in pbar:
        with torch.no_grad():
            gen_kwargs["input_ids"]      = batch["prompt_input_ids"].to(model.device)
            gen_kwargs["attention_mask"] = batch["prompt_attention_mask"].to(model.device)
            out = model.generate(**gen_kwargs)

        dec = tokenizer.batch_decode(out, skip_special_tokens=True)
        preds.extend([eval_model.extract_answer(ds_name, s) for s in dec])
        gold.extend([eval_model.extract_answer(ds_name, s)
              for s in batch["chosen_response_only"]])
        pbar.set_postfix(done=f"{idx}/{n_batches}")
        pbar.refresh()                    # instant progress-bar update

    acc = eval_model.compute_accuracy(ds_name, preds, gold)
    print(f"✓ {ds_name:<15}  Acc: {acc*100:5.2f}%  "
          f"({n_ex} ex, {n_batches} batches, split='{split}')")
    return acc

# 👉 monkey-patch the old helper
eval_model.evaluate = evaluate_tqdm_total
print("✅ Patched eval_model.evaluate with tqdm-progress & HumanEval redirect")

## Experiment settings

Choose a precision for the three training stages and their evaluations. The replay fractions and repeat count are in the final cell.

In [ ]:
from pathlib import Path
import json
import os
import re
import shutil
import subprocess
import sys

from utils import get_local_dir

QUANTIZATION = "8bit"  # "none", "8bit", or "4bit"
NLU_DATASETS = ["boolq", "piqa", "social_i_qa", "arc-challenge",
                "arc-easy", "openbookqa", "hellaswag", "winogrande"]
MATH_DATASETS = ["gsm8k"]
CODE_DATASETS = ["codealpaca"]
BASE_ENV = {"QUIET_SKIP_LOGS": "1", "HYDRA_FULL_ERROR": "1"}
EPOCHS = 1

def hydra_list(items):
    return "[" + ",".join(items) + "]"

def run_stream(cmd, env=None, quiet_patterns=None, summarize=True):
    """
    Run a subprocess and stream stdout in real time, filtering noisy lines.

    Args:
      cmd: list[str] command to run
      env: dict[str,str] extra env vars (merged with os.environ)
      quiet_patterns: list[str|Pattern] extra regexes to suppress
      summarize: whether to print a count of suppressed lines at the end
    """
    # Merge env and force unbuffered output from the child
    env = {**os.environ, "PYTHONUNBUFFERED": "1", **(env or {})}

    # Default noise filters (case-insensitive)
    patterns = [
        # ✅ correct regex: \d+ (digits). This hides the frequent "skipping logging" lines.
        re.compile(r"^skipping logging after \d+ steps to avoid logging too frequently", re.I),
    ]

    # Allow caller to pass extra patterns (strings or compiled regex)
    if quiet_patterns:
        for pat in quiet_patterns:
            patterns.append(re.compile(pat, re.I) if isinstance(pat, str) else pat)

    suppressed = 0
    print("▶", " ".join(map(str, cmd)))
    with subprocess.Popen(
        cmd,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
        env=env,
    ) as p:
        for line in p.stdout:
            l = line.rstrip("\n")
            if any(pat.search(l) for pat in patterns):
                suppressed += 1
                continue
            print(l)
        ret = p.wait()

    if summarize and suppressed:
        print(f"… (suppressed {suppressed} noisy log lines)")
    if ret != 0:
        raise subprocess.CalledProcessError(ret, cmd)

# workspace roots
WORKSPACE = Path("/workspace") if Path("/workspace").exists() else (Path.cwd() / "workspace")
MODELS_ROOT = (WORKSPACE / "models").resolve()
MODELS_ROOT.mkdir(parents=True, exist_ok=True)

def read_base_model_from_adapter(adapter_dir: Path, fallback: str = None) -> str:
    cfg = adapter_dir / "adapter_config.json"
    if not cfg.exists():
        raise FileNotFoundError(f"adapter_config.json not found in {adapter_dir}")
    data = json.loads(cfg.read_text())
    for k in ["base_model_name_or_path", "base_model", "base_model_name"]:
        if k in data and data[k]:
            return data[k]
    if fallback:
        print(f"⚠️ No base model key in adapter_config.json; using fallback: {fallback}")
        return fallback
    raise ValueError("Could not determine base model from adapter_config.json")

def print_tree(path: Path, title: str):
    print(title, "→", path)
    if not path.exists():
        print("  (missing)")
        return
    items = list(sorted(path.iterdir()))
    if not items:
        print("  (empty)")
        return
    for p in items:
        kind = "dir" if p.is_dir() else "file"
        try:
            size = p.stat().st_size if p.is_file() else "-"
        except Exception:
            size = "-"
        print(f" - {p.name:40s} ({kind}, {size})")

## Training stages

Stage A trains on the eight NLU tasks. Stage B adds GSM8K with replay from NLU. Stage C adds CodeAlpaca with replay from math and NLU.

In [ ]:
def train_stage_a():

    STAGE = "stageA_nlu"
    if QUANTIZATION != "none":
        STAGE = f"stageA_nlu_{QUANTIZATION}"

    STAGE_DIR   = (MODELS_ROOT / f"stageA_{QUANTIZATION}").resolve()
    ADAPTER_DST = (STAGE_DIR / "adapter").resolve()
    MERGED_DST  = (STAGE_DIR / "merged").resolve()
    RESULTS_DST = (STAGE_DIR / "results_stageA_peft").resolve()
    for d in [STAGE_DIR, RESULTS_DST]:
        d.mkdir(parents=True, exist_ok=True)

    print("📂 StageA dir:", STAGE_DIR)
    print(f"🔧 Quantization: {QUANTIZATION}")

    # ---- Train (with quantization) ----
    print(f"🚀 Stage A: Training on NLU → {NLU_DATASETS}")
    run_stream([
        sys.executable, "train_lora.py",
        f"exp_name={STAGE}",
        f"datasets={hydra_list(NLU_DATASETS)}",
        "save_every=1000000",
        f"n_epochs={EPOCHS}",
        f"quantization={QUANTIZATION}",  # Apply quantization
        "model.archive=null",
    ], env=BASE_ENV)

    # ---- Find adapter + merged produced by training ----
    print("🔎 Locating adapter & merged artifacts from training…")
    found_adapter = Path(get_local_dir([])) / STAGE
    if not (found_adapter / "adapter_config.json").is_file():
        raise FileNotFoundError(f"Missing adapter in {found_adapter}")
    found_merged = found_adapter / "merged"
    if ADAPTER_DST.exists():
        shutil.rmtree(ADAPTER_DST)
    shutil.copytree(found_adapter, ADAPTER_DST, ignore=shutil.ignore_patterns("merged"))
    print_tree(ADAPTER_DST, "✅ Copied adapter to")

    if QUANTIZATION == "none" and found_merged.is_dir():
        if MERGED_DST.exists():
            shutil.rmtree(MERGED_DST)
        shutil.copytree(found_merged, MERGED_DST)
        print_tree(MERGED_DST, "✅ Copied merged to")
    elif QUANTIZATION != "none":
        print(f"⚠️ Note: Quantized models ({QUANTIZATION}) cannot be merged - using adapter only")

In [ ]:
def train_stage_b(buffer: float):

    STAGE = "stageB_mixed"
    if QUANTIZATION != "none":
        STAGE = f"stageB_mixed_{QUANTIZATION}"

    STAGE_DIR   = (MODELS_ROOT / f"stageB_{QUANTIZATION}").resolve()
    ADAPTER_DST = (STAGE_DIR / "adapter").resolve()
    MERGED_DST  = (STAGE_DIR / "merged").resolve()
    RESULTS_DST = (STAGE_DIR / "results_stageB_peft").resolve()
    for d in [STAGE_DIR, RESULTS_DST]:
        d.mkdir(parents=True, exist_ok=True)

    print("📂 StageB dir:", STAGE_DIR)
    print(f"🔧 Quantization: {QUANTIZATION}")

    # ---- Locate Stage A base ----
    if QUANTIZATION == "none":
        # For non-quantized, prefer merged model
        preferred_stageA = (MODELS_ROOT / f"stageA_{QUANTIZATION}" / "merged").resolve()
        if not preferred_stageA.exists():
            preferred_stageA = (MODELS_ROOT / f"stageA_{QUANTIZATION}" / "adapter").resolve()
    else:
        # For quantized, we need to use base model + adapter path
        preferred_stageA = "meta-llama/Meta-Llama-3.1-8B-Instruct"
        stageA_adapter = (MODELS_ROOT / f"stageA_{QUANTIZATION}" / "adapter").resolve()

    print("🧱 Base for Stage B:", preferred_stageA)

    # ---- Build combined dataset ----
    MIXED_DATASETS = ["gsm8k"]  # 100% math
    if buffer !=0:
        MIXED_DATASETS += [f"{ds}:{buffer}" for ds in NLU_DATASETS]  # 20% of each NLU

    print("🧪 Mixed datasets (WITH shuffling):", MIXED_DATASETS)

    # ---- Train command ----
    train_cmd = [
        sys.executable, "train_lora.py",
        f"exp_name={STAGE}",
        f"datasets={hydra_list(MIXED_DATASETS)}",
        "save_every=1000000",
        f"n_epochs={EPOCHS}",
        f"quantization={QUANTIZATION}",
        f"model.name_or_path={str(preferred_stageA)}",
        "+shuffle=True",
        "lr=5e-5",
        "warmup_steps=100",
    ]

    # For quantized models, load Stage A adapter
    if QUANTIZATION != "none":
        train_cmd.append(f"model.archive={str(stageA_adapter)}")
    else:
        train_cmd.append("model.archive=null")

    print(f"\n🚀 Stage B: Training on combined dataset")
    run_stream(train_cmd, env=BASE_ENV)

    # ---- Find and copy artifacts (same as before) ----
    print("🔎 Locating adapter & merged artifacts from training…")
    found_adapter = Path(get_local_dir([])) / STAGE
    if not (found_adapter / "adapter_config.json").is_file():
        raise FileNotFoundError(f"Missing adapter in {found_adapter}")
    found_merged = found_adapter / "merged"
    if ADAPTER_DST.exists():
        shutil.rmtree(ADAPTER_DST)
    shutil.copytree(found_adapter, ADAPTER_DST, ignore=shutil.ignore_patterns("merged"))
    print_tree(ADAPTER_DST, "✅ Copied adapter to")

    if QUANTIZATION == "none" and found_merged.is_dir():
        if MERGED_DST.exists():
            shutil.rmtree(MERGED_DST)
        shutil.copytree(found_merged, MERGED_DST)
        print_tree(MERGED_DST, "✅ Copied merged to")

In [ ]:
def train_stage_c(buffer: float):

    STAGE = "stageC_mixed"
    if QUANTIZATION != "none":
        STAGE = f"stageC_mixed_{QUANTIZATION}"

    STAGE_DIR   = (MODELS_ROOT / f"stageC_{QUANTIZATION}").resolve()
    ADAPTER_DST = (STAGE_DIR / "adapter").resolve()
    MERGED_DST  = (STAGE_DIR / "merged").resolve()
    RESULTS_DST = (STAGE_DIR / "results_stageC_peft").resolve()
    for d in [STAGE_DIR, RESULTS_DST]:
        d.mkdir(parents=True, exist_ok=True)

    print("📂 StageC dir:", STAGE_DIR)
    print(f"🔧 Quantization: {QUANTIZATION}")

    # ---- Locate Stage B base ----
    if QUANTIZATION == "none":
        # For non-quantized, prefer merged model
        preferred_stageB = (MODELS_ROOT / f"stageB_{QUANTIZATION}" / "merged").resolve()
        if not preferred_stageB.exists():
            preferred_stageB = (MODELS_ROOT / f"stageB_{QUANTIZATION}" / "adapter").resolve()
    else:
        # For quantized, use base model + adapter
        preferred_stageB = "meta-llama/Meta-Llama-3.1-8B-Instruct"
        stageB_adapter = (MODELS_ROOT / f"stageB_{QUANTIZATION}" / "adapter").resolve()

    print("🧱 Base for Stage C:", preferred_stageB)

    # ---- Build combined dataset ----
    MIXED_DATASETS = ["codealpaca"]  # 100% code
    if buffer != 0:
        MIXED_DATASETS += ["gsm8k:"+str(buffer)]  # 20% math
        MIXED_DATASETS += [f"{ds}:{buffer}" for ds in NLU_DATASETS]  # 20% NLU

    print("🧪 Mixed datasets (WITH shuffling):", MIXED_DATASETS)

    # ---- Train command ----
    train_cmd = [
        sys.executable, "train_lora.py",
        f"exp_name={STAGE}",
        f"datasets={hydra_list(MIXED_DATASETS)}",
        "save_every=1000000",
        f"n_epochs={EPOCHS}",
        f"quantization={QUANTIZATION}",
        f"model.name_or_path={str(preferred_stageB)}",
        "+shuffle=True",
        "lr=5e-5",
        "warmup_steps=100",
    ]

    # For quantized models, load Stage B adapter
    if QUANTIZATION != "none":
        train_cmd.append(f"model.archive={str(stageB_adapter)}")
    else:
        train_cmd.append("model.archive=null")

    print(f"\n🚀 Stage C: Training on combined dataset")
    run_stream(train_cmd, env=BASE_ENV)

    # ---- Find and copy artifacts ----
    print("🔎 Locating adapter & merged artifacts from training…")
    found_adapter = Path(get_local_dir([])) / STAGE
    if not (found_adapter / "adapter_config.json").is_file():
        raise FileNotFoundError(f"Missing adapter in {found_adapter}")
    found_merged = found_adapter / "merged"
    if ADAPTER_DST.exists():
        shutil.rmtree(ADAPTER_DST)
    shutil.copytree(found_adapter, ADAPTER_DST, ignore=shutil.ignore_patterns("merged"))
    print_tree(ADAPTER_DST, "✅ Copied adapter to")

    if QUANTIZATION == "none" and found_merged.is_dir():
        if MERGED_DST.exists():
            shutil.rmtree(MERGED_DST)
        shutil.copytree(found_merged, MERGED_DST)
        print_tree(MERGED_DST, "✅ Copied merged to")

    print("\n" + "="*60)
    print(f"🎯 All Stages Complete with {QUANTIZATION} quantization!")
    print("="*60)

## Evaluate a checkpoint

Call `evaluate_stage("stageA")`, `evaluate_stage("stageB")`, or `evaluate_stage("stageC")` after training. Use `evaluate_stage("baseline")` to evaluate the base model at the selected precision.

In [ ]:
def evaluate_stage(stage):
    print(f"📊 Evaluating {stage} with {QUANTIZATION} quantization")

    # Determine model path based on stage and quantization
    if stage == "baseline":
        model_path = "meta-llama/Meta-Llama-3.1-8B-Instruct"
        adapter_path = None
    else:
        stage_dir = MODELS_ROOT / f"{stage}_{QUANTIZATION}"

        if QUANTIZATION == "none":
            # For non-quantized, prefer merged model
            merged_path = stage_dir / "merged"
            if merged_path.exists():
                model_path = str(merged_path)
                adapter_path = None
            else:
                # Fallback to adapter
                adapter_path = str(stage_dir / "adapter")
                # Read base model from adapter config
                model_path = read_base_model_from_adapter(Path(adapter_path),
                                                         fallback="meta-llama/Meta-Llama-3.1-8B-Instruct")
        else:
            # For quantized, always use base + adapter
            adapter_path = str(stage_dir / "adapter")
            model_path = "meta-llama/Meta-Llama-3.1-8B-Instruct"

    print(f"📍 Model: {model_path}")
    print(f"📍 Adapter: {adapter_path if adapter_path else 'None (using merged/base)'}")

    # Create args
    args = argparse.Namespace(
        model_name     = model_path,
        adapter_path   = adapter_path if adapter_path else model_path,
        batch_size     = 32,  # Smaller batch for quantized
        seed           = 0,
        verbose        = False,
        sample         = True,
        results_path   = str(MODELS_ROOT / f"results_{stage}_{QUANTIZATION}"),
        sparsity_ratio = 0.0,
        quantization   = QUANTIZATION,
    )

    # Load model once
    print("Loading model and tokenizer...")
    model, tok = eval_model.load_model_tokenizer(args)

    # Evaluate on all three dataset groups
    results = {}

    print("\n" + "="*60)
    print("📚 Evaluating NLU tasks...")
    print("="*60)
    for task in NLU_DATASETS:
        acc = eval_model.evaluate(task, model, tok, args)
        results[task] = acc * 100

    print("\n" + "="*60)
    print("🔢 Evaluating MATH tasks...")
    print("="*60)
    for task in MATH_DATASETS:
        acc = eval_model.evaluate(task, model, tok, args)
        results[task] = acc * 100

    print("\n" + "="*60)
    print("💻 Evaluating CODE tasks...")
    print("="*60)
    for task in CODE_DATASETS:
        acc = eval_model.evaluate(task, model, tok, args)
        results[task] = acc * 100

    # Summary
    print("\n" + "="*60)
    print(f"📈 {stage.upper()} EVALUATION SUMMARY ({QUANTIZATION})")
    print("="*60)

    # Group results
    nlu_results = {k: v for k, v in results.items() if k in NLU_DATASETS}
    math_results = {k: v for k, v in results.items() if k in MATH_DATASETS}
    code_results = {k: v for k, v in results.items() if k in CODE_DATASETS}

    # Print grouped results
    print(f"\n🧠 NLU Performance:")
    for task, acc in nlu_results.items():
        print(f"  {task:<20} {acc:6.2f}%")
    print(f"  {'Average':<20} {sum(nlu_results.values())/len(nlu_results):6.2f}%")

    print(f"\n🔢 MATH Performance:")
    for task, acc in math_results.items():
        print(f"  {task:<20} {acc:6.2f}%")

    print(f"\n💻 CODE Performance:")
    for task, acc in code_results.items():
        print(f"  {task:<20} {acc:6.2f}%")

    print(f"\n📊 Overall Average: {sum(results.values())/len(results):.2f}%")

## Run the replay sweep

Train Stage A once per repeat, then train and evaluate Stages B and C for each replay fraction. Later fractions reuse the stage output paths.

In [ ]:
for repeat in range(2):
    train_stage_a()
    evaluate_stage("stageA")
    for fraction in [0.20, 0.10, 0.5, 0.01, 0.02, 0.001, 0]:
        print(f"Repeat {repeat + 1}, replay {fraction:.1%}")
        train_stage_b(fraction)
        evaluate_stage("stageB")
        train_stage_c(fraction)
        evaluate_stage("stageC")
        torch.cuda.empty_cache()